In [29]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML

# Import data

In [30]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 0.5, "long_term": 0.1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,0.50,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.49,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.48,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.47,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.46,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [31]:
df_matrix_mf = df.copy()
df_matrix_mf = df_matrix_mf[df["type"] == "top_track"]
df_matrix_mf["username"] = df_matrix_mf["username"].astype("category")
df_matrix_mf["id"] = df_matrix_mf["id"].astype("category")
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100
df_matrix_mf["affinity"]

0        50.0
1        49.0
2        48.0
3        47.0
4        46.0
         ... 
12318     1.0
12319     0.8
12320     0.6
12321     0.4
12322     0.2
Name: affinity, Length: 1200, dtype: float64

In [32]:
# Create the matrix dataset
class MatrixDataset:
    _matrix: np.ndarray

    @property
    def matrix(self) -> np.ndarray:
        return self._matrix

    def __init__(self, num_users: int, num_items: int):
        # A matrix of shape (num_users, num_items)
        self._matrix = np.zeros((num_users, num_items))

    def add_interaction(self, user_id: int, item_id: int, value: float):
        """
        Adds a new interaction to the matrix.
        :param user_id: The user id.
        :param item_id: The item id.
        :param value: The value of the interaction.
        """
        self._matrix[user_id, item_id] = value

    def fill_from_df(self, users: pd.Series, items: pd.Series, values: pd.Series):
        """
        Fills the matrix from a dataframe.
        :param df: The dataframe.
        :param user_col: The user column.
        :param item_col: The item column.
        :param value_col: The value column.
        """
        assert (
            len(users) == len(items) == len(values)
        ), "The length of the users, items and values must be the same."

        for user, item, value in zip(users, items, values):
            self.add_interaction(user, item, value)

    def __str__(self):
        return str(self._matrix)

In [33]:
num_users = len(df_matrix_mf["username"].unique())
num_items = len(df_matrix_mf["id"].unique())

In [34]:
matrix_mf = MatrixDataset(num_users, num_items)
matrix_mf.fill_from_df(df_matrix_mf["username"].cat.codes, df_matrix_mf["id"].cat.codes, df_matrix_mf["affinity"])
R = matrix_mf.matrix
R

array([[ 0. ,  0. , 62. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. , 54. ],
       ...,
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 4.4, 30. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ..., 46. ,  0. ,  0. ]])

In [35]:
def convert_to_ids(values: List[str], column: str) -> List[int]:
    """
    Gets the user ids from the usernames.
    :param usernames: The usernames.
    :return: The user ids.
    """
    return df_matrix_mf[df_matrix_mf[column].isin(values)][column].cat.codes.tolist()

def retrieve_value_from_ids(ids: List[int], column: str) -> str:
    """
    Gets the value from the ids.
    :param ids: The ids.
    :return: The value.
    """
    return df_matrix_mf[df_matrix_mf[column].cat.codes.isin(ids)][column].tolist()

def get_df_rows_from_ids(ids: List[int], column: str, search_in: pd.DataFrame) -> pd.DataFrame:
    """
    Gets the dataframe rows from the ids.
    :param ids: The ids.
    :return: The dataframe rows.
    """
    return search_in[search_in[column].cat.codes.isin(ids)]

In [36]:
# Create a matrix U that contains the index that sorts the users by their affinity
I = np.argsort(matrix_mf.matrix, axis=1)
I.shape

(8, 923)

In [37]:
def compute_alpha(matrix: np.ndarray) -> np.ndarray:
    """
    Computes the alpha matrix.
    :param matrix: The matrix.
    :param k: The number of neighbors.
    :return: The alpha matrix.
    """
    num_items = matrix.shape[1] * matrix.shape[0]
    sum_matrix = matrix.flatten().sum()
    number_of_zeros = num_items - np.count_nonzero(matrix)
    alpha = number_of_zeros / sum_matrix
    return alpha

alpha = compute_alpha(matrix_mf.matrix)
print(alpha)
R *= alpha
R

0.264317361339022


array([[ 0.        ,  0.        , 16.3876764 , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , 14.27313751],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 1.16299639,  7.92952084,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ..., 12.15859862,
         0.        ,  0.        ]])

_Let $l_{u,i}$ denote the event that user $u$ has chosen to interact with item $i$ (user $u$ prefers item $i$). Then, we can let the probability of this event occurring be distributed according to a logistic function parameterized by the sum of the inner product of user and item latent factor vectors and user and item biases._
$$
p(l_{ui} | x_u, y_i, \beta_i, \beta_j) = \frac{\exp(x_uy_i^T + \beta_u + \beta_i)}{1 + \exp(x_uy_i^T + \beta_u + \beta_i)}
$$



The log posterior probability of $x_u, y_i, \beta_u, \beta_i$ given the observed data $\mathbf{R}$ is given by:
$$
\log p(\mathbf{X}, \mathbf{Y}, \beta | \mathbf{R}) = \sum_{u,i} \alpha r_{ui} (x_u^T y_i + \beta_u + \beta_i) - (1 + \alpha r_{ui}) \log(1 + \exp(x_u^T y_i + \beta_u + \beta_i)) - \frac{\lambda}{2} ||x_u||^2 - \frac{\lambda}{2} ||y_i||^2
$$

In [38]:
def log_posterior(
    X: torch.Tensor,
    Y: torch.Tensor,
    beta_u: torch.Tensor,
    beta_i: torch.Tensor,
    R: torch.Tensor,
    alpha: float,
    lambd: float,
) -> torch.Tensor:
    """
    Computes the log posterior of the model, which is:
    :param X: The latent vectors of the users.
    :param Y: The latent vectors of the items.
    :param beta_u: The bias of the users.
    :param beta_i: The bias of the items.
    :param R: The matrix of interactions.
    :param alpha: The alpha parameter.
    :param lambd: The lambda parameter.
    :return: The log posterior.
    """
    # Compute the first term
    term1 = alpha * R * (torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i)
    term2 = (1 + alpha * R) * torch.log1p(torch.exp(torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i))
    regularization = (lambd / 2) * (torch.norm(X, p=2) ** 2 + torch.norm(Y, p=2) ** 2)
    return torch.sum(term1 - term2) - regularization

In [39]:
def mpr(I: torch.Tensor, R: torch.Tensor) -> float:
    """
    Compute the Mean Percentile Ranking (MPR) using a sorted index matrix.

    :param I: Matrix of sorted indices of items for each user.
    :param R: Rating or interaction matrix.
    :return: MPR value.
    """
    num_users, num_items = R.shape
    total_interactions = torch.sum(R)

    # Initialize MPR
    mpr = 0.0

    # Iterate over each user
    for u in range(num_users):
        # Get the indices of the items in sorted order for this user
        sorted_indices = I[u]

        # Calculate the rank for each item
        for i in range(num_items):
            item_index = sorted_indices[i]
            rank = i / num_items  # Percentile rank
            mpr += R[u, item_index] * rank

    # Normalize by the total number of interactions
    mpr /= total_interactions

    return mpr

In [40]:
def compute_predictions(
    X: torch.Tensor,
    Y: torch.Tensor,
    beta_u: torch.Tensor,
    beta_i: torch.Tensor,
) -> torch.Tensor:
    """
    Computes the predictions of the model.
    :param X: The latent vectors of the users.
    :param Y: The latent vectors of the items.
    :param beta_u: The bias of the users.
    :param beta_i: The bias of the items.
    :param R: The matrix of interactions.
    :return: The predictions.
    """
    return torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i

In [41]:
def train(
    num_latent: int,
    lambd: float,
    alpha: float,
    learning_rate: float,
    epochs: int,
):
    """
    Trains the model.
    :param num_latent: The number of latent factors.
    :param lambd: The lambda parameter.
    :param alpha: The alpha parameter.
    :param learning_rate: The learning rate.
    :param epochs: The number of epochs.
    """
    mprs = []  # Record MPRs for each epoch
    losses = []  # Record losses for each epoch

    # Initialize user and item latent factor matrices and bias vectors
    R = torch.tensor(matrix_mf.matrix, dtype=torch.float32)
    X = torch.randn(num_users, num_latent, requires_grad=True)
    Y = torch.randn(num_items, num_latent, requires_grad=True)
    beta_u = torch.randn(num_users, requires_grad=True)
    beta_i = torch.randn(num_items, requires_grad=True)

    # The objective is to minimize the negative log posterior
    optimizer = optim.Adagrad([X, Y, beta_u, beta_i], lr=learning_rate)

    # Training loop
    for epoch in tqdm(range(epochs)):
        # Fix X and B and take a step toward Y and B
        X.requires_grad = False
        Y.requires_grad = True
        beta_u.requires_grad = False
        beta_i.requires_grad = True
        optimizer.zero_grad()
        loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)
        loss.backward()
        optimizer.step()

        # Fix Y and B and take a step toward X and B
        X.requires_grad = True
        Y.requires_grad = False
        beta_u.requires_grad = True
        beta_i.requires_grad = False
        optimizer.zero_grad()
        loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)
        loss.backward()
        optimizer.step()

        # Make predictions
        predictions = compute_predictions(X, Y, beta_u, beta_i)

        # Compute MPR
        I = torch.argsort(predictions, descending=True, dim=1)
        mpr_value = mpr(I, R)

        # Record MPR
        mprs.append(mpr_value)

        # Record loss
        losses.append(loss.item())

        # Print progress
        print(f"Epoch {epoch + 1} - Loss: {loss.item():.4f} - MPR: {mpr_value:.4f}")

    return X, Y, beta_u, beta_i, mprs, losses

In [42]:
num_latent = 100
X, Y, beta_u, beta_i, mprs, losses = train(
    num_latent=num_latent,
    lambd=0.00,
    alpha=alpha,
    learning_rate=0.1,
    epochs=300,
)

  1%|          | 2/300 [00:00<00:17, 16.94it/s]

Epoch 1 - Loss: 22938.0547 - MPR: 0.3467
Epoch 2 - Loss: 13523.7295 - MPR: 0.2499


  1%|▏         | 4/300 [00:00<00:16, 17.76it/s]

Epoch 3 - Loss: 8478.7676 - MPR: 0.1782
Epoch 4 - Loss: 5634.6465 - MPR: 0.1282


  2%|▏         | 6/300 [00:00<00:16, 18.05it/s]

...

Epoch 295 - Loss: 1383.2073 - MPR: 0.0285
Epoch 296 - Loss: 1383.1973 - MPR: 0.0285


 99%|█████████▉| 298/300 [00:16<00:00, 18.69it/s]

Epoch 297 - Loss: 1383.1882 - MPR: 0.0285
Epoch 298 - Loss: 1383.1783 - MPR: 0.0285


100%|██████████| 300/300 [00:16<00:00, 18.40it/s]

Epoch 299 - Loss: 1383.1694 - MPR: 0.0285
Epoch 300 - Loss: 1383.1597 - MPR: 0.0285


In [43]:
px.line(y=mprs, title="MPR over epochs").show()
px.line(y=losses, title="Loss over epochs").show()

In [44]:
user_latent = X.detach().numpy()
item_latent = Y.detach().numpy()

In [45]:
if num_latent <= 3:
    # Add the latent vectors to the dataframe
    df_matrix_mf["user_latent"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: user_latent[x]
    )
    df_matrix_mf["item_latent"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: item_latent[x]
    )

    # Add the biases to the dataframe
    df_matrix_mf["user_bias"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: beta_u[x].item()
    )
    df_matrix_mf["item_bias"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: beta_i[x].item()
    )

    for i in range(num_latent):
        df_matrix_mf[f"user_latent_{i}"] = df_matrix_mf["user_latent"].apply(lambda x: x[i])
        df_matrix_mf[f"latent_{i}"] = df_matrix_mf["item_latent"].apply(lambda x: x[i])

    fig = plotting.plot_latent_space(
        df_matrix_mf,
        color=df_matrix_mf["username"],
        text=df_matrix_mf["username"],
        title="User latent space",
        latent_columns=["latent_0", "latent_1", "latent_2"],
    )
    fig.show()

    df_user_latent = df_matrix_mf.drop_duplicates(subset=["username"])
    fig = plotting.plot_latent_space(
        df_user_latent,
        color=df_user_latent["username"],
        text=df_user_latent["username"],
        title="User latent space",
        latent_columns=["user_latent_0", "user_latent_1", "user_latent_2"],
    )
    fig.show()

In [46]:
def get_recommendations(user_id, X, Y, user_item_matrix, top_n=10):
    """
    Generate top-N item recommendations for a given user.

    :param user_id: The ID of the user for whom to generate recommendations.
    :param X: User factor matrix from Logistic MF.
    :param Y: Item factor matrix from Logistic MF.
    :param user_item_matrix: Original user-item interaction matrix.
    :param top_n: Number of top recommendations to return (default 10).
    :return: List of recommended item IDs, sorted by predicted preference.
    """

    # Check if user_id is valid
    if user_id >= X.shape[0]:
        raise ValueError("User ID is out of range.")

    # Compute predicted scores for all items for this user
    user_vector = X[user_id]
    predicted_scores = np.dot(user_vector, Y.T)

    # Filter out items the user has already interacted with
    known_items = user_item_matrix[user_id].nonzero()[0]
    predicted_scores[known_items] = -np.inf

    # Get indices of top-n items
    recommended_items = np.argsort(predicted_scores)[::-1][:top_n]

    return recommended_items

In [49]:
# Choose a user
user = "michelle"
user_id = convert_to_ids([user], "username")[0]

# Make predictions for the user
predictions = compute_predictions(X, Y, beta_u, beta_i)

# Get the indices of the top 10 recommendations
top_n = 20
recommendations = get_recommendations(user_id, X.detach().numpy(), Y.detach().numpy(), matrix_mf.matrix, top_n=top_n)
recommendations

# Get the top 10 recommendations
top_n_recommendations = retrieve_value_from_ids(recommendations, "id")
top_n_recommendations = get_df_rows_from_ids(recommendations, "id", df_matrix_mf)
top_n_recommendations[spoti.PRETTY_PRINT_FEATURES]

,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
6,jaslkh,ROSALÍA,CHICKEN TERIYAKI,2022,65,0.788,0.4000,0.1150,0.45400,0.000141,0.0686,0.5580,97.936,-6.454,122226,2022,65
26,jaslkh,St Graal,Drag,2023,41,0.666,0.6090,0.0575,0.13900,0.000000,0.0783,0.5840,112.979,-6.580,161333,2023,41
76,jaslkh,Frédéric Chopin,"Chopin: Nocturne No. 20 in C-Sharp Minor, Op. ...",2000,69,0.225,0.0047,0.0509,0.99200,0.881000,0.0722,0.1860,163.665,-30.928,259493,2000,69
90,jaslkh,Gloria Gaynor,I Will Survive,2012,50,0.648,0.6480,0.0444,0.03720,0.000000,0.2380,0.4060,119.270,-11.528,198653,2012,50
112,jaslkh,Fleetwood Mac,Dreams - 2004 Remaster,1977,89,0.828,0.4920,0.0276,0.06440,0.004280,0.1280,0.7890,120.151,-9.744,257800,1977,89
1604,owen,Dinos,93 mesures,2020,57,0.859,0.5290,0.3290,0.04850,0.000000,0.0965,0.3010,95.986,-9.899,265786,2020,57
1625,owen,Dinos,93 mesures,2020,57,0.859,0.5290,0.3290,0.04850,0.000000,0.0965,0.3010,95.986,-9.899,265786,2020,57
6991,brenda,Videoclub,Amour plastique,2021,65,0.696,0.3540,0.0353,0.76700,0.001580,0.1280,0.2420,119.938,-14.043,227026,2021,65
7030,brenda,Videoclub,Amour plastique,2021,65,0.696,0.3540,0.0353,0.76700,0.001580,0.1280,0.2420,119.938,-14.043,227026,2021,65
7109,brenda,Dalida,Gigi l'amoroso,1997,53,0.524,0.4330,0.0631,0.59800,0.000000,0.6150,0.6870,96.994,-8.880,448906,1997,53
